# ✔️ 08_verificacao_sqlserver — Verificação das tabelas no SQL Server
**Squad 2 · Dupla 3** · ferramenta de conferência, **somente leitura** (não grava nada)

Responde a duas perguntas antes da entrega:
1. A tabela `squad2.ecommerce_pedidos` já é a versão nova (só tempo real, gravada pelo `01_carga_sqlserver`) ou ainda a antiga, que misturava o batch?
2. As tabelas da Gold existem no SQL Server com a mesma quantidade de linhas da camada Delta?

| Etapa | Conteúdo |
|---|---|
| 1 | Configuração |
| 2 | Registro da carga em `metadata/` |
| 3 | Impressão digital de `squad2.ecommerce_pedidos` e veredito da versão |
| 4 | Tabelas da Gold: SQL Server × Delta |
| 5 | Resumo da verificação |

Cada etapa **faz** o trabalho e **valida** em seguida: ✅ ok · ⚠️ pendência (segue) · ❌ erro (para).

## Etapa 1 — Configuração
**Faz:** carrega o `00_setup_config` em modo rápido e define o que caracteriza cada versão da tabela.
**Valida:** se o SQL Server responde.

In [0]:
VALIDACAO_COMPLETA = False

In [0]:
%run ./00_setup_config

# ⚙️ 00_setup_config — Configuração central
**Squad 2 · Dupla 3 · `ecommerce_pedidos`**

Prepara endereços, credenciais, conexões e funções usadas por todo o pipeline. **Não lê nem grava pedidos.**

Cada etapa **faz o trabalho e valida em seguida**: ✅ ok · ⚠️ aviso (segue) · ❌ erro (para e explica).

**Uso nos outros notebooks** (duas células; o `%run` fica sozinho):
```
VALIDACAO_COMPLETA = False
```
```
%run ./00_setup_config
```
Dependências em **Environment → Dependencies**: `azure-identity`, `azure-storage-file-datalake`.

## Etapa 1 — Ambiente e endereços
**Faz:** cria a função `checar` e guarda os nomes fixos do projeto.
**Valida:** se está no Databricks e se origem e destino são containers diferentes.

Modo de validação: RÁPIDA
✅ Rodando no Databricks
✅ Origem (raw) e destino (squad2) separados


## Etapa 2 — Credenciais
**Faz:** busca as 7 chaves no Secret Scope. Ficam só na memória (aparecem como `[REDACTED]` se impressas).
**Valida:** se o cofre existe, se nenhuma chave falta e se o host do SQL tem o formato certo.

✅ Cofre 'mercadata-ecommerce-pedidos' encontrado
✅ 7 credenciais carregadas (valores ocultos)
✅ SQL_HOST no formato do Azure SQL


...

✅ url_origem OK
✅ tabela_delta_existe OK (Bronze já existe)
✅ ler_metadata OK (None para anotação inexistente)


## Etapa 6 — Funções do SQL Server
**Faz:** cria as funções de gravar e consultar o banco. Gravação pelo conector `sqlserver` (o `jdbc` é bloqueado no Serverless); leitura pelo `jdbc`.
**Valida:** se toda tabela recebe o schema `squad2`.

✅ Tabelas SQL vão para o schema 'squad2'


## Etapa 7 — Teste completo (só ao rodar este notebook direto)
**Faz e valida:** varre o `raw` · lê o micro-lote mais recente e confere colunas e tipos (regra 1) · grava e relê em `metadata/` · `SELECT 1` no SQL Server · confere se o schema `squad2` existe no banco.
No modo rápido (`%run`), é pulada. No fim, limpa as variáveis temporárias.

Teste completo pulado (modo rápido).
⚙️ 00_setup_config pronto para uso


In [0]:
from pyspark.sql import functions as F

TABELA_SQL = nome_tabela_sql(TABELA)                       # squad2.ecommerce_pedidos
ID_MINIMO_TEMPO_REAL = 1_000_000_000                       # ids do tempo real têm 10 dígitos; os do batch começam em 1
TABELAS_GOLD = ["gold_kpi_volume_pedidos_5min", "gold_kpi_ticket_medio_pagamento", "gold_kpi_taxa_cancelamento_30min",
                "gold_kpi_receita_dia", "gold_hist_volume_pedidos_5min", "gold_hist_ticket_medio_pagamento",
                "gold_hist_taxa_cancelamento_30min", "gold_hist_receita_dia", "gold_alertas"]
verificacoes = []                                           # (item, resultado, situação) para o resumo da Etapa 5

# ------------------------------ validação ------------------------------
try:
    _ok, _erro = ler_sql_query("SELECT 1 AS ok").first()["ok"] == 1, None
except Exception as erro:
    _ok, _erro = False, str(erro)[:200]
checar(_ok, "SQL Server respondeu", f"SQL Server não respondeu: {_erro}")

✅ SQL Server respondeu


## Etapa 2 — Registro da carga
**Faz:** procura a anotação que o `01_carga_sqlserver` grava ao terminar (`metadata/ecommerce_pedidos/carga_sqlserver.json`).
**Valida:** se a carga nova chegou ao fim e se a conferência dela passou.

In [0]:
registro_carga = ler_metadata("carga_sqlserver")

# ------------------------------ validação ------------------------------
checar(registro_carga is not None, f"Carga registrada em {registro_carga and registro_carga.get('executado_em_utc')}",
       "Nenhum registro da carga nova: o 01_carga_sqlserver não rodou até o fim", parar=False)
if registro_carga:
    checar(registro_carga.get("conferencia_ok"), "A conferência da carga passou", "A carga registrou divergência na conferência", parar=False)
verificacoes.append(("Registro da carga em metadata/", registro_carga.get("executado_em_utc") if registro_carga else "ausente",
                     "✅" if registro_carga and registro_carga.get("conferencia_ok") else "⚠️"))

# ------------------------------ resultado ------------------------------
if registro_carga:
    display(pd.DataFrame([{"executado_em_utc": registro_carga.get("executado_em_utc"), "modo": registro_carga.get("modo"),
                           "conferencia_ok": registro_carga.get("conferencia_ok"),
                           **{k: str(v) for k, v in registro_carga.get("impressao_digital", {}).items()}}]))

✅ Carga registrada em 2026-10-08T12:19:25.690821+00:00
✅ A conferência da carga passou


executado_em_utc,modo,conferencia_ok,linhas,pedidos,arquivos,soma_valor_total,inicio,fim
2026-10-08T12:19:25.690821+00:00,overwrite,true,1240,1240,196,330209.5,2026-09-21 11:21:48,2026-10-07 10:55:55


## Etapa 3 — Versão de `squad2.ecommerce_pedidos`
**Faz:** uma consulta com a impressão digital da tabela (linhas, faixa de ids e de datas, e se a coluna `arquivo_origem` existe) e compara com a Bronze.

| Sinal | Versão antiga (misturada) | Versão nova (só tempo real) |
|---|---|---|
| menor `id_pedido` | 1 (ids do batch) | 10 dígitos |
| coluna `arquivo_origem` | não existe | existe |

**Valida:** se a tabela é a versão nova; a quantidade de pedidos é comparada com a Bronze (pode ser menor se chegaram arquivos depois da carga).

In [0]:
try:
    digital = ler_sql_query(f"""
        SELECT COUNT(*) AS linhas, COUNT(DISTINCT id_pedido) AS pedidos, MIN(id_pedido) AS menor_id, MAX(id_pedido) AS maior_id,
               FORMAT(MIN(dt_pedido), 'yyyy-MM-dd HH:mm:ss') AS primeiro_pedido, FORMAT(MAX(dt_pedido), 'yyyy-MM-dd HH:mm:ss') AS ultimo_pedido,
               COL_LENGTH('{TABELA_SQL}', 'arquivo_origem') AS tem_arquivo_origem
        FROM {TABELA_SQL}""").first().asDict()
    _erro = None
except Exception as erro:                                  # tabela inexistente ou sem permissão
    digital, _erro = None, str(erro)[:200]

if digital is None:
    VERSAO = "INEXISTENTE"
elif digital["tem_arquivo_origem"] is not None and (digital["menor_id"] or 0) >= ID_MINIMO_TEMPO_REAL:
    VERSAO = "NOVA (só tempo real)"
else:
    VERSAO = "ANTIGA (mistura batch e tempo real)"

# Referência: quantos pedidos a Bronze tem hoje (a carga espelha o raw do momento em que rodou)
bronze = (ler_delta(CAMINHO_BRONZE).agg(F.count("*").alias("linhas"), F.countDistinct("bronze_source_file").alias("arquivos")).first()
          if tabela_delta_existe(CAMINHO_BRONZE) else None)

# ------------------------------ validação ------------------------------
checar(digital is not None, f"{TABELA_SQL} encontrada", f"{TABELA_SQL} não encontrada: {_erro}", parar=False)
checar(VERSAO.startswith("NOVA"), f"Versão: {VERSAO}", f"Versão: {VERSAO} → rode o 01_carga_sqlserver", parar=False)
if digital and bronze:
    checar(digital["linhas"] <= bronze["linhas"],
           f"{digital['linhas']:,} pedidos no SQL · {bronze['linhas']:,} na Bronze" + (" (chegaram arquivos depois da carga)" if digital["linhas"] < bronze["linhas"] else ""),
           f"SQL com mais linhas ({digital['linhas']:,}) que a Bronze ({bronze['linhas']:,}): sinal de dados de outra origem", parar=False)
verificacoes.append((f"Versão de {TABELA_SQL}", VERSAO, "✅" if VERSAO.startswith("NOVA") else "⚠️"))

# ------------------------------ resultado ------------------------------
if digital:
    display(pd.DataFrame([{k: str(v) for k, v in digital.items()}]))

✅ squad2.ecommerce_pedidos encontrada
✅ Versão: NOVA (só tempo real)
✅ 1,240 pedidos no SQL · 1,240 na Bronze


linhas,pedidos,menor_id,maior_id,primeiro_pedido,ultimo_pedido,tem_arquivo_origem
1240,1240,1790000559,1791381364,2026-09-21 11:21:48,2026-10-07 10:55:55,-1


## Etapa 4 — Tabelas da Gold: SQL Server × Delta
**Faz:** para cada tabela da Gold, conta as linhas no SQL Server e na camada Delta.
**Valida:** tabela presente nos dois lados com a mesma contagem. Os placares (`gold_kpi_...`) são substituídos a cada ciclo; os históricos e o mural de alertas só crescem.

In [0]:
def contar_sql(tabela):
    """Linhas da tabela no SQL Server; None se ela não existe."""
    try:
        return ler_sql_query(f"SELECT COUNT(*) AS n FROM {nome_tabela_sql(tabela)}").first()["n"]
    except Exception:
        return None


gold = []
for tabela in TABELAS_GOLD:
    caminho = caminho_gold(tabela)
    n_delta = ler_delta(caminho).count() if tabela_delta_existe(caminho) else None
    n_sql = contar_sql(tabela)
    situacao = ("✅" if n_delta is not None and n_delta == n_sql else
                "⚠️ ainda não gravada" if n_delta is None else "⚠️ ausente no SQL" if n_sql is None else "⚠️ contagens diferentes")
    gold.append((tabela, n_delta, n_sql, situacao))
gold = pd.DataFrame(gold, columns=["tabela", "linhas_delta", "linhas_sql", "situacao"])
completas = int((gold["situacao"] == "✅").sum())

# ------------------------------ validação ------------------------------
checar(completas == len(gold), f"As {len(gold)} tabelas da Gold estão no SQL Server com a mesma contagem da Delta",
       f"{len(gold) - completas} tabelas com pendência (ver tabela)", parar=False)
verificacoes.append(("Tabelas da Gold no SQL Server", f"{completas} de {len(gold)} conferem", "✅" if completas == len(gold) else "⚠️"))

# ------------------------------ resultado ------------------------------
display(gold.astype({"linhas_delta": "Int64", "linhas_sql": "Int64"}))

✅ As 9 tabelas da Gold estão no SQL Server com a mesma contagem da Delta


tabela,linhas_delta,linhas_sql,situacao
gold_kpi_volume_pedidos_5min,117,117,✅
gold_kpi_ticket_medio_pagamento,4,4,✅
gold_kpi_taxa_cancelamento_30min,4,4,✅
gold_kpi_receita_dia,4,4,✅
gold_hist_volume_pedidos_5min,318,318,✅
gold_hist_ticket_medio_pagamento,136,136,✅
gold_hist_taxa_cancelamento_30min,5,5,✅
gold_hist_receita_dia,34,34,✅
gold_alertas,59,59,✅


## Etapa 5 — Resumo da verificação
**Faz:** junta os resultados das etapas anteriores e indica a próxima ação para cada pendência.

In [0]:
acoes = {
    "Registro da carga em metadata/": "Rodar o 01_carga_sqlserver até o fim",
    f"Versão de {TABELA_SQL}": "Rodar o 01_carga_sqlserver (substitui a tabela antiga)",
    "Tabelas da Gold no SQL Server": "Rodar o 05_ingestao_gold (o espelho completa o que falta)",
}
resumo = pd.DataFrame(verificacoes, columns=["verificacao", "resultado", "situacao"])
resumo["proxima_acao"] = [("—" if s == "✅" else acoes.get(v, "Verificar")) for v, s in zip(resumo["verificacao"], resumo["situacao"])]
resumo["resultado"] = resumo["resultado"].astype(str)          # display() exige um único tipo por coluna

# ------------------------------ validação ------------------------------
pendencias = int((resumo["situacao"] != "✅").sum())
checar(pendencias == 0, "Tudo conferido: SQL Server pronto para a entrega", f"{pendencias} pendências: veja a coluna proxima_acao", parar=False)

# ------------------------------ resultado ------------------------------
display(resumo)

✅ Tudo conferido: SQL Server pronto para a entrega


verificacao,resultado,situacao,proxima_acao
Registro da carga em metadata/,2026-10-08T12:19:25.690821+00:00,✅,—
Versão de squad2.ecommerce_pedidos,NOVA (só tempo real),✅,—
Tabelas da Gold no SQL Server,9 de 9 conferem,✅,—
